In [ ]:
"""
Parte 2 — Melhorando o método ingênuo
Três classes (fundo / interior / fronteira) + mapa de distância ao fundo,
decodificados por watershed marcado. Reaproveita partes da Parte 1
(mask_iou, greedy_match, IOU_THRESHOLDS) — redefinidas aqui para o arquivo
ficar autocontido.

model = UNetTernary()  (já criado por vocês)

ATENÇÃO / PREMISSA A AJUSTAR:
  Como classificação ternária e regressão de distância são duas tarefas
  (multi-task), assumo que UNetTernary.forward(images) devolve uma tupla:
      logits_class : (B, 3, H, W)  -> fundo/interior/fronteira
      dist_pred    : (B, 1, H, W)  -> mapa de distância contínuo, sem ativação
  Se a arquitetura de vocês só tem a cabeça de classes (sem regressão de
  distância), ajustem `forward_pass()` abaixo e removam os termos de
  `dist_pred` na loss e na decodificação (watershed cai de volta a usar
  -prob_interior como elevação, o que também funciona, só que pior).
"""
# Importacao de arquivos
import sys
import os
sys.path.append(os.path.abspath('..')) 
%load_ext autoreload
%autoreload 2
from src import UNetTernary
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from PIL import Image
import cv2
from scipy import ndimage
from skimage.segmentation import watershed
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'xpu' if hasattr(torch, 'xpu') and torch.xpu.is_available() else 'cpu')
IMG_SIZE = 128
model = UNetTernary()
model.to(DEVICE)
path_train = Path('../data/stage1_train')
path_test = Path('../data/stage1_test')
BOUNDARY_THICKNESS = 2  # em pixels, ver justificativa na docstring da função abaixo


# ---------------------------------------------------------------------------
# 1. Geração do rótulo ternário + mapa de distância a partir das instâncias
# ---------------------------------------------------------------------------

def generate_ternary_and_distance(instance_masks, H, W, boundary_thickness=BOUNDARY_THICKNESS):
    """
    Gera, a partir da lista de máscaras binárias de instância (resolução
    original, ANTES do resize):
      - label   (H,W) int64  : 0=fundo, 1=interior, 2=fronteira
      - dist_map(H,W) float32: distância normalizada [0,1] ao fundo

    COMO A FRONTEIRA É GERADA:
      Para cada instância i, erodemos a máscara m_i com elemento estruturante
      de conectividade-8, `boundary_thickness` iterações. O anel de pixels
      removidos pela erosão (m_i AND NOT erode(m_i)) é a fronteira daquela
      instância. A classe "fronteira" global é a UNIÃO desses anéis de todas
      as instâncias, e tem PRIORIDADE sobre "interior": se um pixel cai em
      fronteira de uma instância e interior de outra (instâncias coladas),
      ele fica marcado como fronteira — que é justamente o pixel ambíguo que
      queremos que o modelo aprenda a reconhecer como "corte" entre núcleos.

    ESPESSURA (boundary_thickness=2 px, padrão):
      - Fina demais (1px): depois que a CNN suaviza a predição, o gap de 1px
        some facilmente e o watershed volta a fundir instâncias vizinhas.
      - Grossa demais (4-5px): em núcleos pequenos (diâmetro ~8-10px) a
        erosão come quase todo o interior, deixando poucos pixels de
        marcador confiável para o watershed, e a classe interior fica
        pequena/ruidosa demais para treinar bem.
      - 2px é o meio-termo padrão adotado em trabalhos de segmentação de
        núcleos/células com watershed marcado.

    MAPA DE DISTÂNCIA:
      Para cada instância, calculamos a distância euclidiana ao fundo
      (scipy.ndimage.distance_transform_edt) e normalizamos pelo máximo
      DENTRO daquela própria instância. Isso é importante: sem essa
      normalização por instância, núcleos grandes dominariam a loss (teriam
      valores de distância muito maiores que núcleos pequenos); normalizando
      cada um pelo seu próprio pico, todo núcleo contribui numa escala
      [0,1] comparável, do centro (valor 1) até a borda (valor ~0).
    """
    label = np.zeros((H, W), dtype=np.int64)
    dist_map = np.zeros((H, W), dtype=np.float32)
    struct = ndimage.generate_binary_structure(2, 2)

    interiors, boundaries = [], []
    for m in instance_masks:
        m_bool = m.astype(bool)
        if m_bool.sum() == 0:
            continue
        eroded = ndimage.binary_erosion(m_bool, structure=struct,
                                         iterations=boundary_thickness)
        ring = m_bool & ~eroded
        interiors.append(eroded)
        boundaries.append(ring)

        dt = ndimage.distance_transform_edt(m_bool)
        max_dt = dt.max()
        if max_dt > 0:
            dist_map = np.maximum(dist_map, dt / max_dt)

    for interior in interiors:
        label[interior] = 1
    for ring in boundaries:            # fronteira sobrescreve (prioridade máxima)
        label[ring] = 2

    return label, dist_map


# ---------------------------------------------------------------------------
# 2. Dataset ternário
# ---------------------------------------------------------------------------

class DSB2018TernaryDataset(Dataset):
    def __init__(self, root, img_size=IMG_SIZE, boundary_thickness=BOUNDARY_THICKNESS,
                 augment=None):
        self.root = Path(root)
        self.ids = sorted([p.name for p in self.root.iterdir() if p.is_dir()])
        self.img_size = img_size
        self.boundary_thickness = boundary_thickness
        self.augment = augment

    def __len__(self):
        return len(self.ids)

    def _load_instance_masks(self, mask_dir):
        mask_files = sorted(mask_dir.glob('*.png'))
        return [(np.array(Image.open(mf).convert('L')) > 0).astype(np.uint8)
                for mf in mask_files]

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img_path = self.root / img_id / 'images' / f'{img_id}.png'
        mask_dir = self.root / img_id / 'masks'

        image = np.array(Image.open(img_path).convert('RGB'))
        instance_masks = self._load_instance_masks(mask_dir)
        H, W = image.shape[:2]

        # gera rótulo ternário e mapa de distância NA RESOLUÇÃO ORIGINAL,
        # antes de redimensionar — assim a erosão/distância não é afetada
        # por artefatos de interpolação do resize
        label, dist_map = generate_ternary_and_distance(
            instance_masks, H, W, self.boundary_thickness)

        image_r = cv2.resize(image, (self.img_size, self.img_size),
                              interpolation=cv2.INTER_LINEAR)
        label_r = cv2.resize(label.astype(np.uint8), (self.img_size, self.img_size),
                              interpolation=cv2.INTER_NEAREST).astype(np.int64)
        dist_r = cv2.resize(dist_map, (self.img_size, self.img_size),
                             interpolation=cv2.INTER_LINEAR)

        if self.augment is not None:
            aug = self.augment(image=image_r, masks=[label_r, dist_r])
            image_r, (label_r, dist_r) = aug['image'], aug['masks']

        image_t = torch.from_numpy(image_r / 255.0).permute(2, 0, 1).float()
        label_t = torch.from_numpy(label_r).long()                       # (H,W)
        dist_t = torch.from_numpy(dist_r.astype(np.float32)).unsqueeze(0)  # (1,H,W)

        instance_masks_r = [
            cv2.resize(m, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
            for m in instance_masks
        ]

        return image_t, label_t, dist_t, instance_masks_r, img_id


def collate_fn_ternary(batch):
    images = torch.stack([b[0] for b in batch])
    labels = torch.stack([b[1] for b in batch])
    dists = torch.stack([b[2] for b in batch])
    instance_masks = [b[3] for b in batch]
    ids = [b[4] for b in batch]
    return images, labels, dists, instance_masks, ids


# ---------------------------------------------------------------------------
# 3. Pesos de classe (a fronteira é minoritária) — balanceamento por
#    frequência mediana (median frequency balancing)
# ---------------------------------------------------------------------------

def compute_class_weights(loader, num_classes=3):
    """
    Conta pixels de cada classe no dataset de treino e calcula o peso
    w_c = mediana(freq) / freq_c.
    Classes raras (fronteira) recebem peso > 1; a classe dominante (fundo)
    recebe peso < 1. Isso evita que a CE seja dominada pelo fundo e que a
    fronteira (poucos pixels, mas crítica p/ separar instâncias) seja
    ignorada pelo otimizador.
    """
    counts = np.zeros(num_classes, dtype=np.float64)
    for _, labels, _, _, _ in loader:
        for c in range(num_classes):
            counts[c] += (labels == c).sum().item()
    freq = counts / counts.sum()
    weights = np.median(freq) / np.clip(freq, 1e-12, None)
    return torch.tensor(weights, dtype=torch.float32)


# ---------------------------------------------------------------------------
# 4. Losses
# ---------------------------------------------------------------------------

def focal_loss(logits, targets, weight=None, gamma=2.0):
    """
    CE focal: (1 - p_t)^gamma * CE, com p_t = probabilidade prevista para a
    classe correta. Reduz o peso de pixels "fáceis" (já bem classificados,
    tipicamente fundo) e concentra o gradiente nos pixels difíceis
    (fronteira, interior perto da borda). `weight` (opcional) combina com
    o balanceamento por frequência mediana acima — pode usar os dois juntos.
    """
    logpt = F.log_softmax(logits, dim=1)
    ce = F.nll_loss(logpt, targets, weight=weight, reduction='none')  # (B,H,W)
    pt = logpt.gather(1, targets.unsqueeze(1)).squeeze(1).exp()       # (B,H,W)
    focal_term = (1 - pt).clamp(min=0) ** gamma
    return (focal_term * ce).mean()


def distance_loss(dist_pred, dist_target, loss_type='l1'):
    if loss_type == 'l1':
        return F.l1_loss(dist_pred, dist_target)
    return F.mse_loss(dist_pred, dist_target)


def combined_loss(logits_class, labels, dist_pred, dist_target,
                   class_weight=None, use_focal=True, dist_weight=1.0,
                   dist_loss_type='l1'):
    if use_focal:
        cls_loss = focal_loss(logits_class, labels, weight=class_weight)
    else:
        cls_loss = F.cross_entropy(logits_class, labels, weight=class_weight)
    # d_loss = distance_loss(dist_pred, dist_target, dist_loss_type)
    # return cls_loss + dist_weight * d_loss, cls_loss.item(), d_loss.item()
    if dist_pred is not None:
        d_loss = distance_loss(dist_pred, dist_target, dist_loss_type)
        return cls_loss + dist_weight * d_loss, cls_loss.item(), d_loss.item()
    else:
        return cls_loss, cls_loss.item(), 0.0

# ---------------------------------------------------------------------------
# 5. Forward adapter — AJUSTEM CONFORME A ASSINATURA REAL DE UNetTernary
# ---------------------------------------------------------------------------

def forward_pass(model, images):
    """
    Espera-se que model(images) devolva (logits_class, dist_pred).
    Se a arquitetura só devolver logits_class (sem cabeça de distância),
    troquem por:
        logits_class = model(images)
        dist_pred = None
    e adaptem combined_loss / decode_watershed para ignorar dist_pred.
    """
    # logits_class, dist_pred = model(images)
    logits_class = model(images)
    dist_pred = None
    return logits_class, dist_pred


# ---------------------------------------------------------------------------
# 6. Treino
# ---------------------------------------------------------------------------

def train_model_ternary(model, train_loader, val_loader, class_weight,
                         epochs=30, lr=1e-3, use_focal=True, dist_weight=1.0):
    class_weight = class_weight.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                             factor=0.5, patience=3)
    history = {'train_loss': [], 'val_loss': [], 'val_cls_loss': [], 'val_dist_loss': []}
    best_val = float('inf')

    for epoch in range(epochs):
        model.train()
        running = 0.0
        for images, labels, dists, _, _ in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}'):
            images, labels, dists = images.to(DEVICE), labels.to(DEVICE), dists.to(DEVICE)
            optimizer.zero_grad()
            logits_class, dist_pred = forward_pass(model, images)
            loss, _, _ = combined_loss(logits_class, labels, dist_pred, dists,
                                        class_weight=class_weight, use_focal=use_focal,
                                        dist_weight=dist_weight)
            loss.backward()
            optimizer.step()
            running += loss.item() * images.size(0)
        train_loss = running / len(train_loader.dataset)

        model.eval()
        val_running, cls_running, dist_running = 0.0, 0.0, 0.0
        with torch.no_grad():
            for images, labels, dists, _, _ in val_loader:
                images, labels, dists = images.to(DEVICE), labels.to(DEVICE), dists.to(DEVICE)
                logits_class, dist_pred = forward_pass(model, images)
                loss, cls_l, dist_l = combined_loss(logits_class, labels, dist_pred, dists,
                                                      class_weight=class_weight,
                                                      use_focal=use_focal,
                                                      dist_weight=dist_weight)
                val_running += loss.item() * images.size(0)
                cls_running += cls_l * images.size(0)
                dist_running += dist_l * images.size(0)
        n = len(val_loader.dataset)
        val_loss, cls_loss, d_loss = val_running / n, cls_running / n, dist_running / n
        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_cls_loss'].append(cls_loss)
        history['val_dist_loss'].append(d_loss)
        print(f'  train_loss={train_loss:.4f}  val_loss={val_loss:.4f} '
              f'(cls={cls_loss:.4f}, dist={d_loss:.4f})')

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), 'best_model_ternary.pt')

    return history


# ---------------------------------------------------------------------------
# 7. Decodificação por watershed marcado
# ---------------------------------------------------------------------------

def decode_watershed(class_probs, dist_pred=None, interior_threshold=0.5,
                      fg_threshold=0.5, min_marker_size=5):
    """
    class_probs: (3,H,W) softmax das classes [fundo, interior, fronteira].
    dist_pred:   (H,W) opcional, mapa de distância previsto.

    1. MARCADORES = componentes conexas da classe "interior" binarizada
       (prob_interior > interior_threshold). Cada componente vira a
       semente de uma futura instância — é isto que faz o watershed ser
       "marcado" e não sofrer de over-segmentation like watershed clássico.
    2. MÁSCARA de foreground = tudo que não é fundo (interior + fronteira),
       impede o watershed de "vazar" e invadir o fundo.
    3. SUPERFÍCIE DE ELEVAÇÃO = -dist_pred (se disponível) ou -prob_interior
       como fallback. O watershed "inunda" a partir dos marcadores subindo
       essa superfície; como o centro dos núcleos tem distância maior
       (elevação mais baixa = "vale") e as bordas/fronteiras têm distância
       menor (elevação mais alta = "montanha"), a água de duas instâncias
       vizinhas se encontra exatamente na fronteira entre elas, separando-as.
    """
    prob_fundo, prob_interior, prob_fronteira = class_probs

    markers_bin = prob_interior > interior_threshold
    markers, num_markers = ndimage.label(markers_bin, structure=np.ones((3, 3)))
    for i in range(1, num_markers + 1):
        if (markers == i).sum() < min_marker_size:
            markers[markers == i] = 0

    foreground_mask = (prob_interior + prob_fronteira) > fg_threshold

    elevation = -dist_pred if dist_pred is not None else -prob_interior

    labels_ws = watershed(elevation, markers=markers, mask=foreground_mask)

    instances = []
    for i in range(1, labels_ws.max() + 1):
        inst = (labels_ws == i).astype(np.uint8)
        if inst.sum() > 0:
            instances.append(inst)
    return instances


# ---------------------------------------------------------------------------
# 8. Avaliação de instâncias (reaproveita a MESMA regra de matching da
#    Parte 1 — guloso por IoU decrescente — para os números ficarem
#    comparáveis entre o método ingênuo e este)
# ---------------------------------------------------------------------------

def mask_iou(m1, m2):
    inter = np.logical_and(m1, m2).sum()
    union = np.logical_or(m1, m2).sum()
    return inter / union if union > 0 else 0.0

def greedy_match(pred_masks, gt_masks, iou_threshold):
    n_pred, n_gt = len(pred_masks), len(gt_masks)
    if n_pred == 0 and n_gt == 0:
        return 0, 0, 0
    if n_pred == 0:
        return 0, 0, n_gt
    if n_gt == 0:
        return 0, n_pred, 0
    iou_matrix = np.zeros((n_pred, n_gt))
    for i, pm in enumerate(pred_masks):
        for j, gm in enumerate(gt_masks):
            iou_matrix[i, j] = mask_iou(pm, gm)
    pairs = sorted(((iou_matrix[i, j], i, j) for i in range(n_pred) for j in range(n_gt)),
                    key=lambda x: -x[0])
    matched_pred, matched_gt, tp = set(), set(), 0
    for iou, i, j in pairs:
        if iou < iou_threshold:
            break
        if i in matched_pred or j in matched_gt:
            continue
        matched_pred.add(i); matched_gt.add(j); tp += 1
    return tp, n_pred - tp, n_gt - tp

IOU_THRESHOLDS = np.arange(0.50, 1.00, 0.05)

@torch.no_grad()
def evaluate_instances_watershed(model, loader, interior_threshold=0.5,
                                  fg_threshold=0.5, min_marker_size=5):
    model.eval()
    agg = {t: {'tp': 0, 'fp': 0, 'fn': 0} for t in IOU_THRESHOLDS}
    per_image_records = []

    for images, labels, dists, instance_masks_batch, ids in tqdm(loader, desc='Avaliando (watershed)'):
        images = images.to(DEVICE)
        logits_class, dist_pred = forward_pass(model, images)
        probs = F.softmax(logits_class, dim=1).cpu().numpy()          # (B,3,H,W)
        dist_np = dist_pred.cpu().numpy() if dist_pred is not None else None

        for b in range(images.size(0)):
            class_probs = probs[b]
            d_map = dist_np[b, 0] if dist_np is not None else None
            gt_masks = instance_masks_batch[b]

            pred_masks = decode_watershed(class_probs, d_map,
                                           interior_threshold, fg_threshold,
                                           min_marker_size)

            n_gt, n_pred = len(gt_masks), len(pred_masks)
            count_error = abs(n_pred - n_gt)

            ap_per_threshold = []
            for t in IOU_THRESHOLDS:
                tp, fp, fn = greedy_match(pred_masks, gt_masks, t)
                agg[t]['tp'] += tp; agg[t]['fp'] += fp; agg[t]['fn'] += fn
                denom = tp + fp + fn
                ap_per_threshold.append(tp / denom if denom > 0 else 1.0)

            per_image_records.append({
                'id': ids[b], 'n_gt': n_gt, 'n_pred': n_pred,
                'count_error': count_error,
                'mAP_image': float(np.mean(ap_per_threshold)),
            })

    ap_by_threshold = {}
    for t in IOU_THRESHOLDS:
        tp, fp, fn = agg[t]['tp'], agg[t]['fp'], agg[t]['fn']
        denom = tp + fp + fn
        ap_by_threshold[round(t, 2)] = tp / denom if denom > 0 else 1.0
    mAP = float(np.mean(list(ap_by_threshold.values())))

    df = pd.DataFrame(per_image_records)
    return {
        'ap_by_threshold': ap_by_threshold,
        'mAP': mAP,
        'mean_count_error': df['count_error'].mean(),
        'per_image_df': df,
    }


# ---------------------------------------------------------------------------
# Execução
# ---------------------------------------------------------------------------

if __name__ == '__main__':
    model = UNetTernary()   # já definido por vocês
    model.to(DEVICE)

    full_train_ds = DSB2018TernaryDataset(path_train)
    n = len(full_train_ds)
    n_val = int(0.15 * n)
    n_train = n - n_val
    train_ds, val_ds = torch.utils.data.random_split(
        full_train_ds, [n_train, n_val],
        generator=torch.Generator().manual_seed(42)
    )

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,
                               collate_fn=collate_fn_ternary)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False,
                             collate_fn=collate_fn_ternary)

    print('Calculando pesos de classe (balanceamento por frequência mediana)...')
    # class_weight = compute_class_weights(train_loader)
    class_weight = torch.tensor([0.09790992736816406, 1.0, 1.6255546808242798])
    print(f'Pesos [fundo, interior, fronteira] = {class_weight.tolist()}')

    history = train_model_ternary(model, train_loader, val_loader, class_weight,
                                   epochs=10, lr=1e-3, use_focal=True, dist_weight=1.0)

    model.load_state_dict(torch.load('best_model_ternary.pt'))

    results = evaluate_instances_watershed(model, val_loader)

    print('\nAP por limiar de IoU (watershed):')
    for t, ap in results['ap_by_threshold'].items():
        print(f'  IoU={t:.2f}: AP={ap:.4f}')
    print(f"\nmAP (0.50:0.95): {results['mAP']:.4f}")
    print(f"Erro absoluto médio de contagem por imagem: {results['mean_count_error']:.3f}")

    results['per_image_df'].to_csv('instance_eval_watershed.csv', index=False)

    # comparação direta com o baseline ingênuo (Parte 1), se o CSV existir
    try:
        baseline_df = pd.read_csv('instance_eval_per_image.csv')
        print(f"\nComparação:")
        print(f"  Baseline (limiar+CC): erro médio de contagem = "
              f"{baseline_df['count_error'].mean():.3f}")
        print(f"  Watershed marcado:    erro médio de contagem = "
              f"{results['mean_count_error']:.3f}")
    except FileNotFoundError:
        pass

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Calculando pesos de classe (balanceamento por frequência mediana)...
Pesos [fundo, interior, fronteira] = [0.09790992736816406, 1.0, 1.6255546808242798]


Epoch 1/10: 100%|██████████| 36/36 [08:18<00:00, 13.85s/it]


  train_loss=0.0823  val_loss=0.0600 (cls=0.0600, dist=0.0000)


Epoch 2/10: 100%|██████████| 36/36 [07:47<00:00, 12.99s/it]


  train_loss=0.0473  val_loss=0.0452 (cls=0.0452, dist=0.0000)


Epoch 3/10: 100%|██████████| 36/36 [08:08<00:00, 13.57s/it]


  train_loss=0.0393  val_loss=0.0480 (cls=0.0480, dist=0.0000)


Epoch 4/10:  56%|█████▌    | 20/36 [04:51<03:10, 11.92s/it]